# 서울시 음식점 그래프 알고리즘 분석

## 팀원 및 음식점 정보
- 김수민: 성북구(소테), 강서구(유림닭도리탕)
- 이채영: 은평구(맛있는 칼국수), 마포구(오몬자)
- 진가연: 용산구(남산광어), 성북구(와이인)
- 위주은: 용산구(효뜨), 성북구(문화식당)
- 유가현: 강북구(스페이스), 성북구(르부아)
- 안예원: 강남구(오몬자), 성북구(라라면가)
- 이예지: 성북구(타코스퀘어), 송파구(미엔아이)
- 정세희: 마포구(따로집), 성북구(코지밀)
- 서지윤: 송파구(푸주옥), 강남구(비욘드비엣남)
- 이채민: 성동구(프레고클럽), 용산구(이태원화로)

In [ ]:
# 필요한 라이브러리 설치 및 임포트
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import font_manager, rc
from collections import deque
import numpy as np

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 그래프 시각화를 위한 설정
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['figure.dpi'] = 100

## 1. 음식점 및 이동시간 데이터 정의

In [ ]:
# 음식점 리스트 (모든 음식점)
restaurants = [
    '소테',
    '유림닭도리탕',
    '맛있는 칼국수',
    '오몬자',
    '남산광어',
    '와이인',
    '효뜨',
    '문화식당',
    '스페이스',
    '르부아',
    '라라면가',
    '타코스퀘어',
    '미엔아이',
    '따로집',
    '코지밀',
    '푸주옥',
    '비욘드비엣남',
    '프레고클럽',
    '이태원화로'
]

# 간선 정보 (음식점1, 음식점2, 이동시간)
edges = [
    ('소테', '유림닭도리탕', 37),
    ('유림닭도리탕', '맛있는 칼국수', 14),
    ('맛있는 칼국수', '오몬자', 11),
    ('오몬자', '남산광어', 37),
    ('남산광어', '와이인', 29),
    ('와이인', '효뜨', 21),
    ('효뜨', '문화식당', 36),
    ('문화식당', '스페이스', 22),
    ('스페이스', '르부아', 18),
    ('르부아', '오몬자', 37),
    ('오몬자', '라라면가', 39),
    ('라라면가', '타코스퀘어', 25),
    ('타코스퀘어', '미엔아이', 26),
    ('미엔아이', '따로집', 39),
    ('따로집', '코지밀', 42),
    ('코지밀', '푸주옥', 36),
    ('푸주옥', '비욘드비엣남', 19),
    ('비욘드비엣남', '프레고클럽', 27),
    ('프레고클럽', '이태원화로', 25),
    ('이태원화로', '소테', 31)
]

print(f"총 음식점 수: {len(restaurants)}")
print(f"총 간선 수: {len(edges)}")

## 2. 그래프 생성 및 시각화

In [ ]:
# 그래프 생성
G = nx.Graph()

# 노드 추가
G.add_nodes_from(restaurants)

# 간선 추가 (가중치 포함)
for edge in edges:
    G.add_edge(edge[0], edge[1], weight=edge[2])

# 그래프 레이아웃 설정 (spring layout으로 자연스럽게 배치)
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# 그래프 그리기
plt.figure(figsize=(16, 12))
nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                       node_size=3000, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6)

# 가중치 표시
edge_labels = nx.get_edge_attributes(G, 'weight')
edge_labels = {k: f"{v}분" for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8)

plt.title('Seoul Restaurant Graph', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('restaurant_graph.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"그래프 정보:")
print(f"- 노드(음식점) 수: {G.number_of_nodes()}")
print(f"- 간선 수: {G.number_of_edges()}")
print(f"- 연결 여부: {nx.is_connected(G)}")

## 3. DFS (깊이 우선 탐색) 구현 및 애니메이션

In [ ]:
def dfs_traversal(graph, start_node):
    """DFS 탐색 과정을 단계별로 반환"""
    visited = set()
    stack = [start_node]
    steps = []  # 각 단계를 저장
    
    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            steps.append((node, list(visited).copy(), list(stack).copy()))
            
            # 인접 노드를 스택에 추가 (역순으로)
            neighbors = sorted(list(graph.neighbors(node)), reverse=True)
            for neighbor in neighbors:
                if neighbor not in visited:
                    stack.append(neighbor)
    
    return steps

# DFS 실행
start_restaurant = '소테'
dfs_steps = dfs_traversal(G, start_restaurant)

print(f"DFS 탐색 순서 (시작: {start_restaurant}):")
for i, (node, visited, stack) in enumerate(dfs_steps, 1):
    print(f"{i}. {node}")

In [ ]:
# DFS 애니메이션 생성
def create_dfs_animation():
    fig, ax = plt.subplots(figsize=(16, 12))
    
    def update(frame):
        ax.clear()
        
        if frame >= len(dfs_steps):
            frame = len(dfs_steps) - 1
        
        current_node, visited, stack = dfs_steps[frame]
        
        # 노드 색상 설정
        node_colors = []
        for node in G.nodes():
            if node == current_node:
                node_colors.append('red')  # 현재 방문 중인 노드
            elif node in visited:
                node_colors.append('green')  # 이미 방문한 노드
            else:
                node_colors.append('lightblue')  # 미방문 노드
        
        # 그래프 그리기
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, 
                               node_size=3000, alpha=0.9, ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)
        nx.draw_networkx_edges(G, pos, width=2, alpha=0.4, ax=ax)
        
        # 가중치 표시
        edge_labels = nx.get_edge_attributes(G, 'weight')
        edge_labels = {k: f"{v}분" for k, v in edge_labels.items()}
        nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)
        
        ax.set_title(f'DFS Traversal - Step {frame + 1}/{len(dfs_steps)}\nCurrent: {current_node} | Visited: {len(visited)}', 
                     fontsize=14, fontweight='bold')
        ax.axis('off')
    
    ani = animation.FuncAnimation(fig, update, frames=len(dfs_steps) + 10, 
                                  interval=800, repeat=True)
    
    # MP4로 저장
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=1.5, metadata=dict(artist='Restaurant Graph'), bitrate=1800)
    ani.save('dfs.mp4', writer=writer)
    plt.close()
    
    print("DFS 애니메이션이 'dfs.mp4'로 저장되었습니다.")

create_dfs_animation()

## 4. BFS (너비 우선 탐색) 구현 및 애니메이션

In [ ]:
def bfs_traversal(graph, start_node):
    """BFS 탐색 과정을 단계별로 반환"""
    visited = set()
    queue = deque([start_node])
    steps = []  # 각 단계를 저장
    
    visited.add(start_node)
    
    while queue:
        node = queue.popleft()
        steps.append((node, list(visited).copy(), list(queue).copy()))
        
        # 인접 노드를 큐에 추가
        neighbors = sorted(list(graph.neighbors(node)))
        for neighbor in neighbors:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    
    return steps

# BFS 실행
bfs_steps = bfs_traversal(G, start_restaurant)

print(f"BFS 탐색 순서 (시작: {start_restaurant}):")
for i, (node, visited, queue) in enumerate(bfs_steps, 1):
    print(f"{i}. {node}")

In [ ]:
# BFS 애니메이션 생성
def create_bfs_animation():
    fig, ax = plt.subplots(figsize=(16, 12))
    
    def update(frame):
        ax.clear()
        
        if frame >= len(bfs_steps):
            frame = len(bfs_steps) - 1
        
        current_node, visited, queue = bfs_steps[frame]
        
        # 노드 색상 설정
        node_colors = []
        for node in G.nodes():
            if node == current_node:
                node_colors.append('red')  # 현재 방문 중인 노드
            elif node in visited:
                node_colors.append('green')  # 이미 방문한 노드
            else:
                node_colors.append('lightblue')  # 미방문 노드
        
        # 그래프 그리기
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, 
                               node_size=3000, alpha=0.9, ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)
        nx.draw_networkx_edges(G, pos, width=2, alpha=0.4, ax=ax)
        
        # 가중치 표시
        edge_labels = nx.get_edge_attributes(G, 'weight')
        edge_labels = {k: f"{v}분" for k, v in edge_labels.items()}
        nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)
        
        ax.set_title(f'BFS Traversal - Step {frame + 1}/{len(bfs_steps)}\nCurrent: {current_node} | Visited: {len(visited)}', 
                     fontsize=14, fontweight='bold')
        ax.axis('off')
    
    ani = animation.FuncAnimation(fig, update, frames=len(bfs_steps) + 10, 
                                  interval=800, repeat=True)
    
    # MP4로 저장
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=1.5, metadata=dict(artist='Restaurant Graph'), bitrate=1800)
    ani.save('bfs.mp4', writer=writer)
    plt.close()
    
    print("BFS 애니메이션이 'bfs.mp4'로 저장되었습니다.")

create_bfs_animation()

## 5. Prim 알고리즘 (최소 신장 트리)

In [ ]:
def prim_algorithm(graph, start_node):
    """Prim 알고리즘으로 MST를 생성하고 각 단계를 반환"""
    mst_edges = []
    visited = {start_node}
    steps = []
    
    while len(visited) < len(graph.nodes()):
        min_edge = None
        min_weight = float('inf')
        
        # 방문한 노드에서 미방문 노드로 가는 최소 가중치 간선 찾기
        for node in visited:
            for neighbor in graph.neighbors(node):
                if neighbor not in visited:
                    weight = graph[node][neighbor]['weight']
                    if weight < min_weight:
                        min_weight = weight
                        min_edge = (node, neighbor, weight)
        
        if min_edge:
            mst_edges.append(min_edge)
            visited.add(min_edge[1])
            steps.append((min_edge, list(visited).copy(), mst_edges.copy()))
    
    return mst_edges, steps

# Prim 알고리즘 실행
prim_mst, prim_steps = prim_algorithm(G, start_restaurant)

total_weight = sum(edge[2] for edge in prim_mst)
print(f"\nPrim 알고리즘 결과:")
print(f"- 총 간선 수: {len(prim_mst)}")
print(f"- 총 이동 시간: {total_weight}분")
print(f"\nMST 간선 목록:")
for i, (u, v, w) in enumerate(prim_mst, 1):
    print(f"{i}. {u} - {v}: {w}분")

In [ ]:
# Prim 알고리즘 애니메이션
def create_prim_animation():
    fig, ax = plt.subplots(figsize=(16, 12))
    
    def update(frame):
        ax.clear()
        
        if frame >= len(prim_steps):
            frame = len(prim_steps) - 1
        
        current_edge, visited, mst_edges = prim_steps[frame]
        
        # 노드 색상 설정
        node_colors = []
        for node in G.nodes():
            if node in visited:
                node_colors.append('green')
            else:
                node_colors.append('lightblue')
        
        # 모든 간선 그리기 (회색, 얇게)
        nx.draw_networkx_edges(G, pos, width=1, alpha=0.2, ax=ax)
        
        # MST 간선 그리기 (굵게, 빨간색)
        mst_edge_list = [(e[0], e[1]) for e in mst_edges]
        nx.draw_networkx_edges(G, pos, edgelist=mst_edge_list, 
                               width=3, edge_color='red', ax=ax)
        
        # 현재 추가된 간선 강조 (파란색)
        if current_edge:
            nx.draw_networkx_edges(G, pos, 
                                   edgelist=[(current_edge[0], current_edge[1])], 
                                   width=4, edge_color='blue', ax=ax)
        
        # 노드 그리기
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, 
                               node_size=3000, alpha=0.9, ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)
        
        # 가중치 표시
        edge_labels = nx.get_edge_attributes(G, 'weight')
        edge_labels = {k: f"{v}분" for k, v in edge_labels.items()}
        nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)
        
        current_weight = sum(e[2] for e in mst_edges)
        ax.set_title(f'Prim Algorithm - Step {frame + 1}/{len(prim_steps)}\n' + 
                     f'MST Edges: {len(mst_edges)} | Total Weight: {current_weight}min', 
                     fontsize=14, fontweight='bold')
        ax.axis('off')
    
    ani = animation.FuncAnimation(fig, update, frames=len(prim_steps) + 10, 
                                  interval=1000, repeat=True)
    
    # MP4로 저장
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=1.2, metadata=dict(artist='Restaurant Graph'), bitrate=1800)
    ani.save('prim.mp4', writer=writer)
    plt.close()
    
    print("Prim 알고리즘 애니메이션이 'prim.mp4'로 저장되었습니다.")

create_prim_animation()

## 6. Kruskal 알고리즘 (최소 신장 트리)

In [ ]:
class UnionFind:
    """Union-Find 자료구조 (Kruskal 알고리즘용)"""
    def __init__(self, nodes):
        self.parent = {node: node for node in nodes}
        self.rank = {node: 0 for node in nodes}
    
    def find(self, node):
        if self.parent[node] != node:
            self.parent[node] = self.find(self.parent[node])
        return self.parent[node]
    
    def union(self, node1, node2):
        root1 = self.find(node1)
        root2 = self.find(node2)
        
        if root1 != root2:
            if self.rank[root1] > self.rank[root2]:
                self.parent[root2] = root1
            elif self.rank[root1] < self.rank[root2]:
                self.parent[root1] = root2
            else:
                self.parent[root2] = root1
                self.rank[root1] += 1
            return True
        return False

def kruskal_algorithm(graph):
    """Kruskal 알고리즘으로 MST를 생성하고 각 단계를 반환"""
    # 모든 간선을 가중치 순으로 정렬
    edges_list = [(u, v, data['weight']) for u, v, data in graph.edges(data=True)]
    edges_list.sort(key=lambda x: x[2])
    
    uf = UnionFind(graph.nodes())
    mst_edges = []
    steps = []
    
    for edge in edges_list:
        u, v, weight = edge
        if uf.union(u, v):
            mst_edges.append(edge)
            steps.append((edge, mst_edges.copy(), True))  # 채택됨
        else:
            steps.append((edge, mst_edges.copy(), False))  # 거부됨 (사이클 생성)
        
        if len(mst_edges) == len(graph.nodes()) - 1:
            break
    
    return mst_edges, steps

# Kruskal 알고리즘 실행
kruskal_mst, kruskal_steps = kruskal_algorithm(G)

total_weight_k = sum(edge[2] for edge in kruskal_mst)
print(f"\nKruskal 알고리즘 결과:")
print(f"- 총 간선 수: {len(kruskal_mst)}")
print(f"- 총 이동 시간: {total_weight_k}분")
print(f"\nMST 간선 목록:")
for i, (u, v, w) in enumerate(kruskal_mst, 1):
    print(f"{i}. {u} - {v}: {w}분")

In [ ]:
# Kruskal 알고리즘 애니메이션
def create_kruskal_animation():
    fig, ax = plt.subplots(figsize=(16, 12))
    
    def update(frame):
        ax.clear()
        
        if frame >= len(kruskal_steps):
            frame = len(kruskal_steps) - 1
        
        current_edge, mst_edges, accepted = kruskal_steps[frame]
        
        # 모든 간선 그리기 (회색, 얇게)
        nx.draw_networkx_edges(G, pos, width=1, alpha=0.2, ax=ax)
        
        # MST 간선 그리기 (굵게, 빨간색)
        mst_edge_list = [(e[0], e[1]) for e in mst_edges]
        nx.draw_networkx_edges(G, pos, edgelist=mst_edge_list, 
                               width=3, edge_color='red', ax=ax)
        
        # 현재 검사 중인 간선 강조
        if current_edge:
            color = 'blue' if accepted else 'orange'
            nx.draw_networkx_edges(G, pos, 
                                   edgelist=[(current_edge[0], current_edge[1])], 
                                   width=4, edge_color=color, ax=ax)
        
        # 노드 그리기
        nx.draw_networkx_nodes(G, pos, node_color='lightgreen', 
                               node_size=3000, alpha=0.9, ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)
        
        # 가중치 표시
        edge_labels = nx.get_edge_attributes(G, 'weight')
        edge_labels = {k: f"{v}분" for k, v in edge_labels.items()}
        nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)
        
        current_weight = sum(e[2] for e in mst_edges)
        status = "Accepted" if accepted else "Rejected (Cycle)"
        ax.set_title(f'Kruskal Algorithm - Step {frame + 1}/{len(kruskal_steps)}\n' + 
                     f'Checking: {current_edge[0]} - {current_edge[1]} ({current_edge[2]}min) - {status}\n' +
                     f'MST Edges: {len(mst_edges)} | Total Weight: {current_weight}min', 
                     fontsize=14, fontweight='bold')
        ax.axis('off')
    
    ani = animation.FuncAnimation(fig, update, frames=len(kruskal_steps) + 10, 
                                  interval=1000, repeat=True)
    
    # MP4로 저장
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=1.2, metadata=dict(artist='Restaurant Graph'), bitrate=1800)
    ani.save('kruskal.mp4', writer=writer)
    plt.close()
    
    print("Kruskal 알고리즘 애니메이션이 'kruskal.mp4'로 저장되었습니다.")

create_kruskal_animation()

## 7. 결과 요약 및 비교

In [ ]:
print("\n" + "="*60)
print("결과 요약")
print("="*60)

print(f"\n1. 그래프 정보:")
print(f"   - 총 음식점(노드) 수: {G.number_of_nodes()}")
print(f"   - 총 간선 수: {G.number_of_edges()}")
print(f"   - 연결 그래프 여부: {nx.is_connected(G)}")

print(f"\n2. 탐색 알고리즘:")
print(f"   - DFS 방문 순서: {len(dfs_steps)}개 노드")
print(f"   - BFS 방문 순서: {len(bfs_steps)}개 노드")

print(f"\n3. 최소 신장 트리(MST):")
print(f"   - Prim 알고리즘 총 이동시간: {sum(e[2] for e in prim_mst)}분")
print(f"   - Kruskal 알고리즘 총 이동시간: {sum(e[2] for e in kruskal_mst)}분")
print(f"   - MST 간선 수: {len(prim_mst)}")

print(f"\n4. 생성된 파일:")
print(f"   - restaurant_graph.png: 전체 그래프 시각화")
print(f"   - dfs.mp4: DFS 탐색 애니메이션")
print(f"   - bfs.mp4: BFS 탐색 애니메이션")
print(f"   - prim.mp4: Prim 알고리즘 애니메이션")
print(f"   - kruskal.mp4: Kruskal 알고리즘 애니메이션")

print("\n" + "="*60)
print("모든 과제가 완료되었습니다!")
print("="*60)